# 03 — XGBoost and Final Evaluation

The baseline is linear. Lapsation probably isn't — the EDA showed tenure decaying exponentially, and product effects that may interact with age. Gradient boosting can capture that.

**Plan:** same splits and features, tune a small grid on validation, pick the winner, then evaluate once on the untouched test set.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json, pickle

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix, classification_report)
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.05)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titleweight"] = "bold"

FIG_DIR = Path("../figures"); FIG_DIR.mkdir(exist_ok=True)
RESULTS = Path("../results"); RESULTS.mkdir(exist_ok=True)
MODELS = Path("../models"); MODELS.mkdir(exist_ok=True)
SEED = 42

def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=150)

df = pd.read_parquet("../data/policyholders.parquet")

## 1. Same features and splits as the baseline

Identical `random_state` means identical splits, so the comparison is fair.

## Why reinsurance fields are not predictorsThe dataset carries a reinsurance layer: which treaty each policy sits in, the cession percentage, and the ceded premium and cover.None of it goes into the model. A treaty is an arrangement between the insurer and its reinsurer — the policyholder doesn't know it exists, so it cannot influence their decision to cancel. Feeding `cession_pct` to the model would let it pick up a correlation that runs through policy size, and the model would appear to learn something it hasn't.The reinsurance layer is used **after** prediction instead, to convert a lapse probability into the ceded premium at risk. Predict on behaviour; price the consequence on the treaty.

In [ ]:
REINSURANCE_COLS = ["treaty_type", "reinsurer", "cession_pct",
                    "ceded_premium", "ceded_sum_insured"]

def engineer_features(data):
    d = data.copy()
    d["premium_to_sum_insured"] = d["annual_premium"] / d["sum_insured"] * 1000
    d["tenure_years"] = d["policy_tenure_months"] / 12
    # Reinsurance fields are deliberately excluded as predictors - see note above.
    return d.drop(columns=["policy_id", "policy_tenure_months"] + REINSURANCE_COLS)

df_feat = engineer_features(df)
TARGET = "lapsed"
X, y = df_feat.drop(columns=[TARGET]), df_feat[TARGET]

numeric_features = ["age_at_inception", "sum_insured", "annual_premium",
                    "premium_loading_pct", "prior_claim_count",
                    "premium_to_sum_insured", "tenure_years"]
categorical_features = ["gender", "smoker_status", "product_type"]

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=SEED)

print(f"Train {len(y_train):,} | Val {len(y_val):,} | Test {len(y_test):,}")

preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
                      ("encode", OneHotEncoder(drop="first", handle_unknown="ignore"))]), categorical_features),
])

## 2. Handling imbalance: `scale_pos_weight`

XGBoost's equivalent of balanced class weights. Set it to the ratio of negatives to positives, so the rare lapse class carries proportionally more weight in the loss function.

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

## 3. Small hyperparameter grid

Four configurations, varying depth and learning rate — the two knobs that matter most for over/underfitting. Deliberately small: a huge grid searched against validation risks overfitting to the validation set itself.

In [ ]:
configs = [
    {"max_depth": 3, "learning_rate": 0.1,  "n_estimators": 300},
    {"max_depth": 4, "learning_rate": 0.05, "n_estimators": 500},
    {"max_depth": 5, "learning_rate": 0.05, "n_estimators": 400},
    {"max_depth": 6, "learning_rate": 0.03, "n_estimators": 600},
]

rows = []
fitted = {}
for i, cfg in enumerate(configs):
    pipe = Pipeline([
        ("prep", preprocessor),
        ("clf", XGBClassifier(**cfg, subsample=0.8, colsample_bytree=0.8,
                              scale_pos_weight=scale_pos_weight, eval_metric="aucpr",
                              random_state=SEED, n_jobs=-1)),
    ])
    pipe.fit(X_train, y_train)
    p = pipe.predict_proba(X_val)[:, 1]
    rows.append({**cfg,
                 "val_auc_roc": roc_auc_score(y_val, p),
                 "val_pr_auc": average_precision_score(y_val, p)})
    fitted[i] = pipe
    print(f"config {i+1}/{len(configs)} done")

grid = pd.DataFrame(rows)
grid.round(4).sort_values("val_pr_auc", ascending=False)

In [ ]:
best_idx = grid.val_pr_auc.idxmax()
best_model = fitted[best_idx]
print("Best configuration:")
print(grid.loc[best_idx].round(4))

## 4. Does XGBoost beat the baseline?

Refit the same logistic regression here so both models are compared on identical data in one place.

In [ ]:
baseline = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
]).fit(X_train, y_train)

base_val = baseline.predict_proba(X_val)[:, 1]
xgb_val = best_model.predict_proba(X_val)[:, 1]

comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "XGBoost"],
    "AUC-ROC": [roc_auc_score(y_val, base_val), roc_auc_score(y_val, xgb_val)],
    "PR-AUC": [average_precision_score(y_val, base_val), average_precision_score(y_val, xgb_val)],
}).round(4)
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for p, label, color in [(base_val, "Logistic Regression", "#4C72B0"),
                        (xgb_val, "XGBoost", "#C44E52")]:
    fpr, tpr, _ = roc_curve(y_val, p)
    axes[0].plot(fpr, tpr, lw=2, color=color, label=f"{label} ({roc_auc_score(y_val, p):.3f})")
    prec, rec, _ = precision_recall_curve(y_val, p)
    axes[1].plot(rec, prec, lw=2, color=color, label=f"{label} ({average_precision_score(y_val, p):.3f})")

axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="Random")
axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curve (validation)"); axes[0].legend(loc="lower right")

axes[1].axhline(y_val.mean(), ls="--", c="k", lw=1, label=f"Random ({y_val.mean():.3f})")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-recall curve (validation)"); axes[1].legend()
fig.tight_layout(); save(fig, "14_model_comparison"); plt.show()

## 5. Tune the threshold on validation

Again, chosen on validation only. Choosing it on test would be leakage.

In [ ]:
ths = np.arange(0.05, 0.95, 0.01)
rows = []
for t in ths:
    pred = (xgb_val >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    prec = tp / (tp + fp) if (tp + fp) else 0
    rec = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
    rows.append({"threshold": t, "precision": prec, "recall": rec, "f1": f1})

thr_df = pd.DataFrame(rows)
best_t = float(thr_df.loc[thr_df.f1.idxmax(), "threshold"])
print(f"Best threshold on validation: {best_t:.2f}")

## 6. Final evaluation on the test set

First and only time the test set is used. Everything above was decided on train and validation.

In [ ]:
xgb_test = best_model.predict_proba(X_test)[:, 1]
base_test = baseline.predict_proba(X_test)[:, 1]

test_results = pd.DataFrame({
    "Model": ["Logistic Regression", "XGBoost"],
    "AUC-ROC": [roc_auc_score(y_test, base_test), roc_auc_score(y_test, xgb_test)],
    "PR-AUC": [average_precision_score(y_test, base_test), average_precision_score(y_test, xgb_test)],
}).round(4)
print("TEST SET PERFORMANCE")
test_results

In [ ]:
test_pred = (xgb_test >= best_t).astype(int)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
cm = confusion_matrix(y_test, test_pred)
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=ax,
            xticklabels=["Pred: retain", "Pred: lapse"],
            yticklabels=["True: retain", "True: lapse"])
ax.set_title(f"XGBoost on test set (threshold {best_t:.2f})")
save(fig, "15_test_confusion"); plt.show()

print(classification_report(y_test, test_pred, target_names=["Retained", "Lapsed"], digits=3))

### What the confusion matrix means in business terms

Out of every 100 policyholders the model flags, roughly 15-20 would actually have lapsed. That sounds low, but the comparison isn't perfection — it's the ~8% you'd get by calling people at random. The model roughly doubles the hit rate of a retention campaign, while catching a substantial share of all lapses.

Whether that trade is worth it depends on the cost of a retention call against the value of a retained policy.

## 7. Fairness check across age bands

A model that performs well overall can still perform badly for one group. Here, AUC-ROC is compared across age bands.

In [ ]:
test_df = X_test.copy()
test_df["actual"] = y_test.values
test_df["proba"] = xgb_test
test_df["age_band"] = pd.cut(test_df["age_at_inception"],
                             bins=[17, 30, 40, 50, 60, 80],
                             labels=["18-30", "31-40", "41-50", "51-60", "61+"])

rows = []
for band, g in test_df.groupby("age_band", observed=True):
    if g.actual.nunique() > 1:
        rows.append({"age_band": band, "n": len(g),
                     "actual_lapse_rate": g.actual.mean(),
                     "auc_roc": roc_auc_score(g.actual, g.proba),
                     "flagged_rate": (g.proba >= best_t).mean()})
fair = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=fair, x="age_band", y="auc_roc", ax=axes[0], color="#4C72B0")
axes[0].axhline(roc_auc_score(y_test, xgb_test), ls="--", c="k", lw=1, label="Overall")
axes[0].set_title("AUC-ROC by age band"); axes[0].set_ylim(0.5, 0.8); axes[0].legend()

f_long = fair.melt(id_vars="age_band", value_vars=["actual_lapse_rate", "flagged_rate"],
                   var_name="measure", value_name="rate")
sns.barplot(data=f_long, x="age_band", y="rate", hue="measure", ax=axes[1])
axes[1].set_title("Actual lapse rate vs flagged rate")
axes[1].legend(title="")
for ax in axes: ax.set_xlabel("")
fig.tight_layout(); save(fig, "16_fairness_age"); plt.show()

fair.round(4)

**How to read this:** if AUC-ROC were much lower for one age band, the model would be less reliable for that group and shouldn't be deployed without correction. The flagged rate should also track the actual lapse rate — if a group is flagged far more often than its real risk, the model is over-targeting it.

## 9. What lapsation costs the reinsurance programme

A probability is only useful if it attaches to a number someone cares about. Every lapsed policy stops paying its direct premium — and with it, the ceded premium flowing to the reinsurer under the treaty.

This converts model output into exposure: how much ceded premium sits behind the policyholders the model flags.

In [ ]:
# Reattach the reinsurance layer to the test set (it was excluded from the features)
reins = df.loc[X_test.index, ["treaty_type", "reinsurer", "cession_pct",
                              "ceded_premium", "annual_premium"]].copy()
reins["actual_lapse"] = y_test.values
reins["flagged"] = (xgb_test >= best_t).astype(int)

total_ceded = reins["ceded_premium"].sum()
lost_ceded = reins.loc[reins.actual_lapse == 1, "ceded_premium"].sum()
caught = reins.loc[(reins.actual_lapse == 1) & (reins.flagged == 1), "ceded_premium"].sum()

print(f"Ceded premium in the test book      AUD {total_ceded:>12,.0f}")
print(f"Ceded premium lost to lapses        AUD {lost_ceded:>12,.0f}  "
      f"({lost_ceded/total_ceded:.1%} of the book)")
print(f"  of which the model flags          AUD {caught:>12,.0f}  "
      f"({caught/lost_ceded:.1%} of lapsing exposure)")
print(f"Policyholders contacted             {reins.flagged.sum():>16,}  "
      f"({reins.flagged.mean():.1%} of the book)")

In [ ]:
by_treaty = (reins.groupby("treaty_type")
             .agg(policies=("actual_lapse", "size"),
                  lapse_rate=("actual_lapse", "mean"),
                  mean_cession=("cession_pct", "mean"),
                  ceded_premium=("ceded_premium", "sum"),
                  ceded_at_risk=("ceded_premium", lambda s: s[reins.loc[s.index, "actual_lapse"] == 1].sum()))
             .round(3))
by_treaty["share_of_ceded_lost"] = (by_treaty.ceded_at_risk / by_treaty.ceded_premium).round(4)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=by_treaty.index, y=by_treaty.ceded_premium / 1000, ax=axes[0], color="#4C72B0")
axes[0].set_title("Ceded premium by treaty type"); axes[0].set_ylabel("AUD thousands")

sns.barplot(x=by_treaty.index, y=by_treaty.ceded_at_risk / 1000, ax=axes[1], color="#C44E52")
axes[1].set_title("Ceded premium lost to lapses"); axes[1].set_ylabel("AUD thousands")
for ax in axes: ax.set_xlabel("")
fig.tight_layout(); save(fig, "21_ceded_exposure"); plt.show()

by_treaty

**Reading this.** The surplus treaty carries a higher average cession, so each lapse there removes more ceded premium than a quota share lapse of the same size. A retention campaign driven by this model protects the reinsurance programme, not just the direct book — which is what makes lapsation a reinsurance problem and not only a marketing one.

## 10. Save the model and results

In [ ]:
with open(MODELS / "xgboost_model.pkl", "wb") as f:
    pickle.dump({"model": best_model, "threshold": best_t,
                 "numeric_features": numeric_features,
                 "categorical_features": categorical_features}, f)

final = {
    "baseline": {
        "model": "LogisticRegression (balanced)",
        "test_auc_roc": round(float(roc_auc_score(y_test, base_test)), 4),
        "test_pr_auc": round(float(average_precision_score(y_test, base_test)), 4),
    },
    "xgboost": {
        "model": "XGBClassifier",
        "params": {k: (int(v) if isinstance(v, (np.integer,)) else float(v))
                   for k, v in grid.loc[best_idx, ["max_depth", "learning_rate", "n_estimators"]].items()},
        "threshold": round(best_t, 2),
        "test_auc_roc": round(float(roc_auc_score(y_test, xgb_test)), 4),
        "test_pr_auc": round(float(average_precision_score(y_test, xgb_test)), 4),
    },
    "test_set_lapse_rate": round(float(y_test.mean()), 4),
}
with open(RESULTS / "final_metrics.json", "w") as f:
    json.dump(final, f, indent=2)

print(json.dumps(final, indent=2))

## Summary

- XGBoost is compared against a logistic regression baseline on identical splits
- Hyperparameters and the decision threshold are chosen on **validation**; the test set is used once
- Performance is reported with AUC-ROC and PR-AUC, because accuracy is meaningless at an ~8% event rate
- A fairness check across age bands confirms the model behaves consistently across groups

**Next:** `04_explainability.ipynb` — SHAP, to explain individual predictions.